# Quinta lista de ejercicios – Selección de variables y generalización
Introducción al Machine Learning · Universidad El Bosque · 2026-2

In [ ]:
import sys
from pathlib import Path

# Permite importar el paquete src/ desde notebooks/
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from src.config import DIR_FIGURAS, DIR_TABLAS
from src.data import cargar_datos, particionar
from src.models import ajustar_y_evaluar, riesgo_en_prueba
from src.plots import graficar_forward
from src.selection import (
    mejor_forward, ranking_correlacion, seleccion_correlacion,
    seleccion_forward, seleccion_pruebas_t,
)

DIR_TABLAS.mkdir(parents=True, exist_ok=True)
DIR_FIGURAS.mkdir(parents=True, exist_ok=True)

## 0. Datos y partición (`random_state=31`)

In [ ]:
X, y = cargar_datos()
p = particionar(X, y)
todas = list(p.X_train.columns)
print(f"Predictoras ({len(todas)}): {todas}")
print(f"Entrenamiento: {len(p.X_train)} | Validación: {len(p.X_val)} | Prueba: {len(p.X_test)}")

## Ejercicio 1. Modelo completo

In [ ]:
_, r1_tr, r1_val = ajustar_y_evaluar(p.X_train, p.y_train, p.X_val, p.y_val)
print(f"Riesgo entrenamiento: {r1_tr:.4f}")
print(f"Riesgo validación   : {r1_val:.4f}")

## Ejercicio 2. Selección estadística de variables
### (a) Correlación con `Salary` (solo entrenamiento)

In [ ]:
ranking = ranking_correlacion(p.X_train, p.y_train)
tabla_corr = ranking.rename("Correlacion").to_frame()
tabla_corr["Correlacion_abs"] = tabla_corr["Correlacion"].abs()
tabla_corr.to_csv(DIR_TABLAS / "ej2a_correlaciones.csv")
tabla_corr.round(4)

### (b) Top 7 por correlación

In [ ]:
top7 = seleccion_correlacion(p.X_train, p.y_train)
_, r2_tr, r2_val = ajustar_y_evaluar(p.X_train[top7], p.y_train, p.X_val[top7], p.y_val)
print("Variables:", top7)
print(f"Riesgo train: {r2_tr:.4f} | valid: {r2_val:.4f}")

### (c) Pruebas t (p < 0.05) en la regresión completa

In [ ]:
vars_t = seleccion_pruebas_t(p.X_train, p.y_train)
_, r3_tr, r3_val = ajustar_y_evaluar(p.X_train[vars_t], p.y_train, p.X_val[vars_t], p.y_val)
print("Variables:", vars_t)
print(f"Riesgo train: {r3_tr:.4f} | valid: {r3_val:.4f}")

### (d) Comparación

In [ ]:
comp2 = pd.DataFrame({
    "Modelo": ["Completo (19)", "Top 7 correlación", "Pruebas t (p<0.05)"],
    "N_variables": [len(todas), len(top7), len(vars_t)],
    "Riesgo_train": [r1_tr, r2_tr, r3_tr],
    "Riesgo_valid": [r1_val, r2_val, r3_val],
})
comp2.to_csv(DIR_TABLAS / "ej2d_comparacion.csv", index=False)
comp2.round(4)

**Análisis:** _escribe aquí tu comparación de los tres modelos._

## Ejercicio 3. Selección hacia adelante
### (a) Tabla

In [ ]:
tabla_fwd = seleccion_forward(p.X_train, p.y_train, p.X_val, p.y_val)
tabla_fwd.to_csv(DIR_TABLAS / "ej3a_forward.csv", index=False)
tabla_fwd.round(4)

### (b) Gráfica

In [ ]:
graficar_forward(tabla_fwd, DIR_FIGURAS / "forward_selection.png")
plt.show()

### (c) Mejor modelo por validación

In [ ]:
vars_fwd, r4_tr, r4_val = mejor_forward(tabla_fwd)
print(f"{len(vars_fwd)} variables: {vars_fwd}")
print(f"Riesgo train: {r4_tr:.4f} | valid: {r4_val:.4f}")

## Ejercicio 4. Comparación final

In [ ]:
modelos = {
    "Todas las variables": todas,
    "Mayor correlación": top7,
    "Pruebas t": vars_t,
    "Selección hacia adelante": vars_fwd,
}
r_train = [r1_tr, r2_tr, r3_tr, r4_tr]
r_val   = [r1_val, r2_val, r3_val, r4_val]
r_test  = [riesgo_en_prueba(p.X_train, p.y_train, p.X_test, p.y_test, v)
           for v in modelos.values()]
nombres = list(modelos)

i_val = min(range(4), key=r_val.__getitem__)
i_tr  = min(range(4), key=r_train.__getitem__)
i_te  = min(range(4), key=r_test.__getitem__)

# (a)
print(f"(a) Modelo elegido: {nombres[i_val]} (riesgo de validación = {r_val[i_val]:.4f})")

In [ ]:
# (b)
tabla_final = pd.DataFrame({
    "Modelo": nombres,
    "Variables": [len(v) for v in modelos.values()],
    "Riesgo_train": r_train,
    "Riesgo_valid": r_val,
    "Riesgo_prueba": r_test,
})
tabla_final.to_csv(DIR_TABLAS / "ej4b_tabla_final.csv", index=False)
tabla_final.round(4)

In [ ]:
# (c)
print(f"Menor riesgo de entrenamiento: {nombres[i_tr]} ({r_train[i_tr]:.4f})")
print(f"Menor riesgo de prueba       : {nombres[i_te]} ({r_test[i_te]:.4f})")
print("¿Coinciden?", "Sí" if i_tr == i_te else "No")

# (d)
print(f"\nElegido por validación: {nombres[i_val]} "
      f"(valid = {r_val[i_val]:.4f}, prueba = {r_test[i_val]:.4f})")
print(f"Menor en prueba       : {nombres[i_te]} (prueba = {r_test[i_te]:.4f})")
print("¿El elegido por validación fue el mejor en prueba?", "Sí" if i_val == i_te else "No")

**Conclusiones:** _escribe aquí tus respuestas a (c) y (d) citando los valores de la tabla._